# AURA — Data Generation

Generate training episodes from the corridor environment + synthetic audio.
Run this on Colab or locally. Saves NPZ files to Google Drive (Colab) or local disk.

In [ ]:
# Clone repo (Colab only)
import os
if 'COLAB_GPU' in os.environ:
    !git clone https://github.com/SotoAlt/aura.git
    %cd aura
    !pip install -q -r world_model/requirements-colab.txt

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/aura/data/train'
else:
    DATA_DIR = 'data/train'

In [ ]:
import numpy as np
from pathlib import Path
from world_model.data.generate import EpisodeGenerator, NPZDataset

## Generate Episodes

| Episodes | Size | Time (M3 MacBook) |
|----------|------|--------------------|
| 1,000    | ~29 MB  | ~5 min |
| 10,000   | ~300 MB | ~50 min |
| 50,000   | ~1.4 GB | ~4.2 hr |

In [ ]:
NUM_EPISODES = 10000  # Adjust as needed
STEPS_PER_EPISODE = 100
SEED = 42

gen = EpisodeGenerator(steps_per_episode=STEPS_PER_EPISODE)
gen.generate_dataset(NUM_EPISODES, DATA_DIR, base_seed=SEED)
print(f'Generated {NUM_EPISODES} episodes to {DATA_DIR}')

## Verify Data

In [ ]:
# Check file count and sizes
data_path = Path(DATA_DIR)
npz_files = sorted(data_path.glob('episode_*.npz'))
print(f'Episodes: {len(npz_files)}')

total_bytes = sum(f.stat().st_size for f in npz_files)
print(f'Total size: {total_bytes / 1e6:.1f} MB')
print(f'Avg per episode: {total_bytes / len(npz_files) / 1e3:.1f} KB')

In [ ]:
# Inspect a single episode
ep = np.load(npz_files[0])
for key in ep.files:
    print(f'  {key}: shape={ep[key].shape}, dtype={ep[key].dtype}')

# Context vector stats
ctx = ep['context']
print(f'\nContext range: [{ctx.min():.3f}, {ctx.max():.3f}]')
print(f'Context mean: {ctx.mean():.3f}')

In [ ]:
# Display sample frames
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
images = ep['image']
for i, ax in enumerate(axes):
    idx = i * (len(images) // 5)
    ax.imshow(images[idx])
    ax.set_title(f'Frame {idx}')
    ax.axis('off')
plt.suptitle('Sample Corridor Frames')
plt.tight_layout()
plt.show()

In [ ]:
# Test batch loading
ds = NPZDataset(DATA_DIR, seq_length=10, batch_size=4)
batch = ds.sample_batch()
print(f'Batch shapes:')
for k, v in batch.items():
    print(f'  {k}: {v.shape} ({v.dtype})')
print('\nData ready for training!')